# Gold Layer: Review Themes

Produces one row per neighbourhood per city summarising the dominant review themes
derived from keyword matching across guest comments.

Joins `{CITY}_REVIEWS` to `{CITY}_LISTINGS` on `listing_id = id` to obtain
`neighbourhood_cleansed`, then explodes `comments_list` to one row per comment
before aggregating keyword counts per neighbourhood.

**Output:** `AIRBNB_INVESTMENT_DB.GOLD.REVIEW_THEMES`

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
CITIES = ['BRISTOL', 'LONDON', 'GREATER_MANCHESTER']

DB         = 'AIRBNB_INVESTMENT_DB'
SCHEMA_SIL = 'SILVER'
SCHEMA_GLD = 'GOLD'
TABLE_OUT  = 'REVIEW_THEMES'

PRICE_KEYWORDS       = 'cheap|expensive|value|price|cost|affordable|overpriced'
CLEANLINESS_KEYWORDS = 'clean|dirty|spotless|tidy|mess|dusty|filthy'
LOCATION_KEYWORDS    = 'location|central|transport|walk|nearby|tube|bus|station'
CHECKIN_KEYWORDS     = 'check-in|checkin|key|access|arrival|lockbox|host|welcome'

In [ ]:
def build_themes(session, city, city_filter):
    import ast

    df_reviews = session.sql(f"""
        SELECT r.LISTING_ID, r.COMMENTS_LIST,
               l.NEIGHBOURHOOD AS neighbourhood_cleansed
        FROM AIRBNB_INVESTMENT_DB.SILVER.REVIEWS_CLEANED r
        JOIN AIRBNB_INVESTMENT_DB.SILVER.LISTINGS_CLEANED l
          ON r.LISTING_ID = l.LISTING_ID
        WHERE r.COMMENTS_LIST IS NOT NULL
        AND (UPPER(l._FILENAME) LIKE '%/{city_filter}/%'
           OR UPPER(l._FILENAME) LIKE '%{city_filter}%')
    """).to_pandas()

    if len(df_reviews) == 0:
        # Fallback: try individual COMMENTS column
        df_reviews = session.sql(f"""
            SELECT r.LISTING_ID, r.COMMENTS AS comment,
                   l.NEIGHBOURHOOD AS neighbourhood_cleansed
            FROM AIRBNB_INVESTMENT_DB.SILVER.REVIEWS_CLEANED r
            JOIN AIRBNB_INVESTMENT_DB.SILVER.LISTINGS_CLEANED l
              ON r.LISTING_ID = l.LISTING_ID
            WHERE r.COMMENTS IS NOT NULL
            AND l._FILENAME LIKE '%{city_filter}%'
        """).to_pandas()
        df = df_reviews.copy()
    else:
        df_reviews['COMMENTS_LIST'] = df_reviews['COMMENTS_LIST'].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
        df = df_reviews.explode('COMMENTS_LIST').rename(
            columns={'COMMENTS_LIST': 'comment'}
        )

    df = df[df['comment'].notna() & (df['comment'].str.strip() != '')]
    df['comment'] = df['comment'].str.lower()

    df['mentions_price']       = df['comment'].str.contains(PRICE_KEYWORDS, regex=True, na=False)
    df['mentions_cleanliness'] = df['comment'].str.contains(CLEANLINESS_KEYWORDS, regex=True, na=False)
    df['mentions_location']    = df['comment'].str.contains(LOCATION_KEYWORDS, regex=True, na=False)
    df['mentions_checkin']     = df['comment'].str.contains(CHECKIN_KEYWORDS, regex=True, na=False)

    df_themes = df.groupby('neighbourhood_cleansed').agg(
        total_reviews              = ('comment', 'count'),
        mentions_price_count       = ('mentions_price', 'sum'),
        mentions_cleanliness_count = ('mentions_cleanliness', 'sum'),
        mentions_location_count    = ('mentions_location', 'sum'),
        mentions_checkin_count     = ('mentions_checkin', 'sum'),
    ).reset_index()

    for col in ['mentions_price_count', 'mentions_cleanliness_count',
                'mentions_location_count', 'mentions_checkin_count']:
        df_themes[col] = df_themes[col].astype(int)

    safe_total = df_themes['total_reviews'].replace(0, float('nan'))
    df_themes['pct_mentions_price']       = (df_themes['mentions_price_count']       / safe_total * 100).round(2)
    df_themes['pct_mentions_cleanliness'] = (df_themes['mentions_cleanliness_count'] / safe_total * 100).round(2)
    df_themes['pct_mentions_location']    = (df_themes['mentions_location_count']    / safe_total * 100).round(2)
    df_themes['pct_mentions_checkin']     = (df_themes['mentions_checkin_count']     / safe_total * 100).round(2)

    count_cols = [
        'mentions_price_count', 'mentions_cleanliness_count',
        'mentions_location_count', 'mentions_checkin_count',
    ]
    label_map = {
        'mentions_price_count':       'price',
        'mentions_cleanliness_count': 'cleanliness',
        'mentions_location_count':    'location',
        'mentions_checkin_count':     'checkin',
    }
    df_themes['top_theme'] = df_themes[count_cols].idxmax(axis=1).map(label_map)

    # Join VADER sentiment scores from GOLD.LISTING_SENTIMENT
    df_sentiment = session.sql(f"""
        SELECT ls.listing_id, ls.avg_sentiment_score,
               l.NEIGHBOURHOOD AS neighbourhood_cleansed
        FROM AIRBNB_INVESTMENT_DB.GOLD.LISTING_SENTIMENT ls
        JOIN AIRBNB_INVESTMENT_DB.SILVER.LISTINGS_CLEANED l
          ON ls.listing_id = l.LISTING_ID
        WHERE l._FILENAME LIKE '%{city_filter}%'
    """).to_pandas()

    if len(df_sentiment) > 0:
        df_sent_agg = df_sentiment.groupby('neighbourhood_cleansed').agg(
            avg_sentiment_score=('avg_sentiment_score', 'mean')
        ).reset_index()
        df_themes = df_themes.merge(df_sent_agg, on='neighbourhood_cleansed', how='left')
        df_themes['avg_sentiment_score'] = df_themes['avg_sentiment_score'].round(4)
    else:
        df_themes['avg_sentiment_score'] = None

    df_themes['computed_at'] = pd.Timestamp.now()

    print(f'Review themes computed for {len(df_themes)} neighbourhoods from {len(df)} comments')
    return df_themes


def write_gold(session, df):
    session.write_pandas(
        df,
        'REVIEW_THEMES',
        database='AIRBNB_INVESTMENT_DB',
        schema='GOLD',
        overwrite=False
    )
    print(f'Written {len(df)} rows to GOLD.REVIEW_THEMES')


def validate(session):
    df_val = session.sql('''
        SELECT city, COUNT(*) AS neighbourhood_count,
               SUM(total_reviews) AS total_reviews,
               COUNT(top_theme) AS non_null_top_theme
        FROM AIRBNB_INVESTMENT_DB.GOLD.REVIEW_THEMES
        GROUP BY city ORDER BY city
    ''').to_pandas()
    print(df_val)

In [ ]:
# --- Orchestration ---
CITY_FILENAME_MAP = {
    'BRISTOL':            'bristol',
    'LONDON':             'london',
    'GREATER_MANCHESTER': 'greater_manchester',
}

try:
    session.sql("""
        TRUNCATE TABLE AIRBNB_INVESTMENT_DB.GOLD.REVIEW_THEMES
    """).collect()
    print('REVIEW_THEMES truncated')
except:
    print('REVIEW_THEMES does not exist yet — will be created')

for city in CITIES:
    print(f'\nProcessing {city}...')
    city_filter = CITY_FILENAME_MAP[city]
    df_themes = build_themes(session, city, city_filter)
    df_themes['city'] = city
    write_gold(session, df_themes)
    print(f'  {city} complete — {len(df_themes)} rows written')

validate(session)

In [ ]:
df_val = session.sql('''
    SELECT city, COUNT(*) AS neighbourhood_count,
           SUM(total_reviews) AS total_reviews,
           COUNT(top_theme) AS non_null_top_theme
    FROM AIRBNB_INVESTMENT_DB.GOLD.REVIEW_THEMES
    GROUP BY city ORDER BY city
''').to_pandas()
print(df_val)

## Review Themes Complete

`GOLD.REVIEW_THEMES` is ready for Streamlit consumption.
Each row represents one neighbourhood per city with keyword-based review theme counts.